# 텍스트 및 이미지 임베딩

**건강 & 피트니스** 임베딩 노트북에 오신 것을 환영합니다!  
이 튜토리얼에서는 다음과 같은 내용을 다룹니다:

1. `EmbeddingsClient`를 **초기화**하여 Microsoft Foundry 프로젝트에 접근합니다.  
2. 건강 테마의 문구를 활용하여 `azure-ai-inference`로 **텍스트를 임베딩**합니다.  

그럼 건강한 아이디어와 함께 재미있게 시작해봅시다! 🍏

> **안내문**: 이 노트북은 교육 목적의 예제입니다. 의학적 조언이 필요할 경우 반드시 전문가와 상담하세요.


## 1. 설정 및 환경 구성

#### 사전 준비 사항:
- Microsoft Foundry에 텍스트 임베딩 모델(**text-embedding-3-small**)을 배포하세요.

#

라이브러리를 임포트하고 다음과 같은 환경 변수를 불러옵니다:
- `ENDPOINT`: 프로젝트 연결 문자열
- `TEXT_EMBEDDING_MODEL`: 텍스트 임베딩 모델 배포 이름

라이브러리를 임포트하고, 환경 변수를 불러온 뒤 `EmbeddingsClient`를 생성합니다.


In [ ]:
import os
from dotenv import load_dotenv
from pathlib import Path

from azure.core.credentials import AzureKeyCredential

from azure.ai.inference import EmbeddingsClient

# Load environment variables
notebook_path = Path().absolute()
parent_dir = notebook_path.parent
load_dotenv(parent_dir / '.env')

# Retrieve from environment
endpoint = os.environ.get("ENDPOINT")
api_key = os.environ.get("API_KEY")
embedding_model = os.environ.get("TEXT_EMBEDDING_MODEL")

# Initialize project client
try:
    embeddings_client = EmbeddingsClient(
        endpoint=endpoint,
        credential=AzureKeyCredential(api_key),
        model=embedding_model
    )
    print("🎉 Successfully created AIProjectClient")
except Exception as e:
    print("❌ Error creating AIProjectClient:", e)

## 2. 텍스트 임베딩

`AzureOpenAI`에서 `embeddings()`를 호출하여 임베딩 클라이언트를 가져옵니다.  
그런 다음, 아래와 같은 건강 테마의 재미있는 문구들을 임베딩해볼 것입니다:

- "🍎 하루에 사과 하나면 의사가 필요 없다"
- "🏋️ 15분 고강도 인터벌 트레이닝 루틴"
- "🧘 마음을 가다듬는 호흡 운동"

출력 결과는 각 문장을 의미론적 공간(semantic space)에서 표현하는 **숫자 벡터**입니다.  
이제 임베딩 결과를 확인해봅시다!


In [ ]:
text_phrases = [
    "An apple a day keeps the doctor away 🍎",
    "Quick 15-minute HIIT workout routine 🏋️",
    "Mindful breathing exercises 🧘"
]

try:
    response = embeddings_client.embed(
        input=text_phrases
    )

    for item in response.data:
        length = len(item.embedding)
        print(
            f"data[{item.index}]: length={length}, "
            f"[{item.embedding[0]}, {item.embedding[1]}, "
            f"..., {item.embedding[length-2]}, {item.embedding[length-1]}]"
        )
    print(response.usage)
except Exception as e:
    print("❌ Error embedding text:", e)

## 3. 프롬프트 템플릿 예제

이번 노트북의 초점은 임베딩이지만, 사용자 메시지에 앞서 컨텍스트를 추가하는 방법도 함께 소개합니다.  
예를 들어, 사용자의 텍스트를 임베딩하기 전에  
“당신은 HealthFitGPT, 피트니스 가이드를 제공하는 모델입니다…”  
와 같은 시스템 프롬프트를 먼저 추가하고 싶을 수 있습니다.  

이러한 작은 설정이 더 **컨텍스트를 인식하는 임베딩**을 만드는 데 도움이 됩니다.

In [ ]:
# A basic prompt template (system-style) we'll prepend to user text.
TEMPLATE_SYSTEM = (
    "You are HealthFitGPT, a fitness guidance model.\n"
    "Please focus on healthy advice and disclaim you're not a doctor.\n\n"
    "User message:"  # We'll append the user message after this.
)

def embed_with_template(user_text):
    """Embed user text with a system template in front."""
    content = TEMPLATE_SYSTEM + " " + user_text

    response = embeddings_client.embed(
        input=[content]
    )

    return response.data[0].embedding

sample_user_text = "Can you suggest a quick home workout for busy moms?"
embedding_result = embed_with_template(sample_user_text)
print("Embedding length:", len(embedding_result))
print("First few dims:", embedding_result[:8])